STANDARD HEADER FOR ALL TRAINING NOTEBOOKS<br>
Every notebook that needs packages starts with this exact block. Copy it identically.

In [ ]:
import subprocess, sys, os

WHEELS = "/kaggle/input/datasets/mahmoodavaram/riva-wheels/trl"
# Install from local wheels, no index, no internet
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-index", "--find-links", WHEELS,
    "trl", "transformers", "accelerate", "datasets",
    "peft", "bitsandbytes"
], check=True, capture_output=True)

os.environ["WANDB_DISABLED"]       = "true"
os.environ["HF_DATASETS_OFFLINE"]  = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_CACHE"]    = "/kaggle/working/cache"
os.makedirs("/kaggle/working/cache", exist_ok=True)

# Paths — match your existing structure exactly
BASE_DATA   = "/kaggle/input/datasets/mahmoodavaram/riva-training-datasets"
BASE_MODEL  = "/kaggle/input/models/mahmoodavaram/qwen2-5-7b/transformers/default/1/qwen2.5-7b"
INSTRUCT_MODEL = "/kaggle/input/models/mahmoodavaram/qwen2-5-7b-instruct/transformers/default/1/qwen2.5-7b-instruct"
PRM_MODEL   = "/kaggle/input/models/mahmoodavaram/qwen2-5-math-prm-7b/transformers/default/1/qwen2.5-math-prm-7b"
EVAL_BASE   = "/kaggle/input/datasets/mahmoodavaram/riva-master-dataset/test"
OUT         = "/kaggle/working"

All output goes to /kaggle/working. At the end of each notebook, Kaggle saves everything in /kaggle/working as output. You then attach the previous notebook's output as an input dataset to the next notebook. This is how checkpoints chain through the pipeline without internet.

THE EVAL FUNCTION BLOCK<br>
Copy this block verbatim into every training notebook. Run it after every saved checkpoint. Never modify it — your baselines depend on this exact implementation.

In [ ]:
import re, json, time
from collections import defaultdict
from datasets import load_from_disk
import shutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_eval_sets():
    for name in ["gsm8k", "mmlu", "strategyqa"]:
        src = f"{EVAL_BASE}/{name}"
        dst = f"{OUT}/eval/{name}"
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
    gsm8k_eval      = load_from_disk(f"{OUT}/eval/gsm8k")
    mmlu_eval       = load_from_disk(f"{OUT}/eval/mmlu")
    strategyqa_eval = load_from_disk(f"{OUT}/eval/strategyqa")
    return gsm8k_eval, mmlu_eval, strategyqa_eval

GSM8K_FEWSHOT = """Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. #### 39

Question: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give Denny?
Answer: Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. #### 8

Question: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?
Answer: Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. #### 9

Question: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?
Answer: Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 = 8 dollars left. #### 8

"""

MMLU_FEWSHOT_TEMPLATE = """The following are multiple choice questions (with answers) about {subject}.

Question: Glucose is transported into the muscle cell:
A) via protein transporters called GLUT4.
B) only in the presence of insulin.
C) via hexokinase.
D) via monocarboxylate transporters.
Answer: A

Question: What is the embryological origin of the hyoid bone?
A) The first pharyngeal arch
B) The first and second pharyngeal arches
C) The second pharyngeal arch
D) The second and third pharyngeal arches
Answer: D

Question: In a genetic test of a newborn, a rare genetic disorder is found that has X-linked recessive transmission. Which of the following statements is likely true?
A) The newborn is a carrier
B) The newborn is unaffected
C) The mother is a carrier
D) The father is affected
Answer: C

Question: A 77-year-old man presents with headache...
Answer: C

Question: A 44-year-old man comes to the office because of a 3-day history of sore throat...
Answer: C

"""

STRATEGYQA_FEWSHOT = """Q: Do hamsters provide food for any animals?
A: Hamsters are prey animals. Prey are food for predators. Thus, hamsters provide food for some animals. Yes

Q: Could Brooke Shields succeed at the 1992 Olympics?
A: Brooke Shields was born on May 31, 1965. The 1992 Olympics were in Barcelona. Brooke Shields would have been 27 at the time. 27 year old can compete in the Olympics. Yes

Q: Hydrogen's atomic number squared exceeds number of Spice Girls?
A: Hydrogen has an atomic number of 1. 1 squared is 1. There are 5 Spice Girls. 1 does not exceed 5. No

Q: Is it common to see frost during some college commencements?
A: College commencement ceremonies can happen in December, May, and June. December is in the winter, so there can be frost. Thus, frost can be common at some college commencements. Yes

Q: Could a llama birth twice during War in Vietnam (1945-46)?
A: The War in Vietnam was 6 months. The gestation period of a llama is 11 months. Thus, a llama could not give birth twice during the War in Vietnam. No

Q: Would a pear sink in water?
A: The density of a pear is about 0.6 g/cm3, which is less than water. Objects less dense than water float. Thus, a pear would not sink. No

"""

GSM_BATCH   = 8
MMLU_BATCH  = 32
STRAT_BATCH = 16

def normalize_number(s):
    s = str(s).strip().rstrip(".,;:")
    s = s.replace(",", "").replace("$", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return s

def extract_gsm(response):
    if "####" in response:
        raw = response.split("####")[1].strip().split("\n")[0]
        return normalize_number(raw.split()[0] if raw.split() else raw)
    nums = re.findall(r"-?\$?[\d,]+\.?\d*", response)
    return normalize_number(nums[-1]) if nums else ""

def extract_yesno(text):
    m = re.search(r"\b(Yes|No)\b", text, re.IGNORECASE)
    return m.group(1).capitalize() if m else None

def make_generate_fn(model, tokenizer):
    tokenizer.padding_side = "left"
    def generate_batch(prompts, max_new_tokens=512):
        lengths = [len(tokenizer.encode(p, add_special_tokens=False)) for p in prompts]
        order   = sorted(range(len(prompts)), key=lambda i: lengths[i], reverse=True)
        sorted_prompts = [prompts[i] for i in order]
        inputs = tokenizer(
            sorted_prompts, return_tensors="pt",
            truncation=True, max_length=3072,
            padding=True, padding_side="left"
        ).to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        prompt_lengths = inputs["input_ids"].shape[1]
        decoded_sorted = [
            tokenizer.decode(outputs[i][prompt_lengths:], skip_special_tokens=True).strip()
            for i in range(len(sorted_prompts))
        ]
        result = [None] * len(prompts)
        for sorted_idx, orig_idx in enumerate(order):
            result[orig_idx] = decoded_sorted[sorted_idx]
        return result
    return generate_batch

def run_full_eval(model, tokenizer, phase_name):
    gsm8k_eval, mmlu_eval, strategyqa_eval = load_eval_sets()
    generate_batch = make_generate_fn(model, tokenizer)
    model.eval()
    
    # GSM8K
    gsm_correct, gsm_n = 0, len(gsm8k_eval)
    for bs in range(0, gsm_n, GSM_BATCH):
        batch = gsm8k_eval.select(range(bs, min(bs+GSM_BATCH, gsm_n)))
        prompts = [GSM8K_FEWSHOT + f"Question: {ex['prompt']}\nAnswer:" for ex in batch]
        responses = generate_batch(prompts, max_new_tokens=512)
        for ex, resp in zip(batch, responses):
            if extract_gsm(resp) == normalize_number(ex["gold_answer"]):
                gsm_correct += 1
    gsm_score = round(gsm_correct / gsm_n * 100, 1)
    
    # MMLU
    import random
    rng = random.Random(42)
    subj_to_idx = defaultdict(list)
    for idx, ex in enumerate(mmlu_eval):
        subj_to_idx[ex["subject"]].append(idx)
    sel = []
    for subj, indices in sorted(subj_to_idx.items()):
        sel.extend(rng.sample(indices, min(10, len(indices))))
    mmlu_correct, mmlu_n = 0, len(sel)
    for bs in range(0, mmlu_n, MMLU_BATCH):
        batch = [mmlu_eval[i] for i in sel[bs:bs+MMLU_BATCH]]
        prompts = []
        for ex in batch:
            subj = ex.get("subject", "general knowledge").replace("_", " ")
            prompts.append(MMLU_FEWSHOT_TEMPLATE.format(subject=subj) + f"Question: {ex['prompt']}\nAnswer:")
        responses = generate_batch(prompts, max_new_tokens=5)
        for ex, resp in zip(batch, responses):
            m = re.search(r"^([ABCD])", resp.strip()) or re.search(r"\b([ABCD])\b", resp)
            pred = m.group(1) if m else ""
            if pred == ex["gold_answer"]:
                mmlu_correct += 1
    mmlu_score = round(mmlu_correct / mmlu_n * 100, 1)
    
    # StrategyQA
    sqa_correct, sqa_n = 0, len(strategyqa_eval)
    subset = strategyqa_eval.shuffle(seed=42)
    for bs in range(0, sqa_n, STRAT_BATCH):
        batch = subset.select(range(bs, min(bs+STRAT_BATCH, sqa_n)))
        prompts = [STRATEGYQA_FEWSHOT + f"Q: {ex['prompt']}\nA:" for ex in batch]
        responses = generate_batch(prompts, max_new_tokens=128)
        for ex, resp in zip(batch, responses):
            lines = [l.strip() for l in resp.strip().split("\n") if l.strip()]
            pred = extract_yesno(lines[-1] if lines else "") or extract_yesno(resp) or ""
            if pred == ex["gold_answer"]:
                sqa_correct += 1
    sqa_score = round(sqa_correct / sqa_n * 100, 1)
    
    avg = round((gsm_score + mmlu_score + sqa_score) / 3, 1)
    results = {
        "phase": phase_name,
        "timestamp": time.strftime("%Y-%m-%d %H:%M"),
        "scores": {"gsm8k": gsm_score, "mmlu": mmlu_score, "strategyqa": sqa_score, "average": avg}
    }
    with open(f"{OUT}/{phase_name}_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n{'='*50}")
    print(f"EVAL: {phase_name}")
    print(f"GSM8K:      {gsm_score}%")
    print(f"MMLU:       {mmlu_score}%")
    print(f"StrategyQA: {sqa_score}%")
    print(f"Average:    {avg}%")
    print(f"{'='*50}")
    return results

NOTEBOOK 1 — DATASET PREPARATION<br>
Type: CPU notebook, no GPU, internet OFF<br>
Input datasets: riva-training-datasets<br>
Output: saved to /kaggle/working, then published as dataset riva-split-datasets<br>
This notebook runs once. It partitions all training data, assigns train/eval roles, and saves clean arrow files that allsubsequent notebooks read from.
What the notebook does:

In [ ]:
import os
import json
import random
from collections import defaultdict
import datasets
from datasets import load_from_disk

# ── Force memory processing and prevent any read-only disk writes ─────
datasets.disable_caching()

random.seed(42)

BASE = "/kaggle/input/datasets/mahmoodavaram/riva-training-datasets"
OUT  = "/kaggle/working"

# ── Inspect actual structure first ────────────────────────────────────
print("=== FILE STRUCTURE ===")
for name in ["gsm8k", "mmlu", "strategyqa", "openmath", "commonsenseqa",
             "hotpotqa", "natural_questions"]:
    path = f"{BASE}/{name}"
    if os.path.isdir(path):
        contents = os.listdir(path)
        print(f"{name}/: {sorted(contents)}")
print()

# ═══════════════════════════════════════════════════════════════════════
# GSM8K
# ═══════════════════════════════════════════════════════════════════════
print("Loading GSM8K...")
gsm_train_ds = load_from_disk(f"{BASE}/gsm8k/train")
print(f"  gsm8k/train rows: {len(gsm_train_ds)}")
print(f"  columns: {gsm_train_ds.column_names}")
print(f"  sample: {gsm_train_ds[0]}")

# Fixed: Forcing in-memory evaluation avoids creating temp files in /kaggle/input
gsm_shuffled    = gsm_train_ds.shuffle(seed=42, keep_in_memory=True)
gsm_pear_source = gsm_shuffled.select(range(3000), keep_in_memory=True)
gsm_grpo_source = gsm_shuffled.select(range(3000, len(gsm_shuffled)), keep_in_memory=True)

gsm_pear_source.save_to_disk(f"{OUT}/gsm_pear_source")
gsm_grpo_source.save_to_disk(f"{OUT}/gsm_grpo_source")
print(f"  GSM8K PEAR: {len(gsm_pear_source)}, GRPO: {len(gsm_grpo_source)}")

# ═══════════════════════════════════════════════════════════════════════
# MMLU
# ═══════════════════════════════════════════════════════════════════════
print("\nLoading MMLU validation split...")
mmlu_val = load_from_disk(f"{BASE}/mmlu/validation")
print(f"  mmlu/validation rows: {len(mmlu_val)}")
print(f"  columns: {mmlu_val.column_names}")
print(f"  sample: {mmlu_val[0]}")

sample = mmlu_val[0]
subj_field = None
for candidate in ['subject', 'Subject', 'topic', 'category']:
    if candidate in sample:
        subj_field = candidate
        break
if subj_field is None:
    print("  WARNING: no subject field found — using all validation rows as-is")

if subj_field:
    by_subj = defaultdict(list)
    for i, row in enumerate(mmlu_val):
        by_subj[row[subj_field]].append(i)
    
    print(f"  Subjects found: {len(by_subj)}")
    for s, v in sorted(by_subj.items(), key=lambda x: len(x[1])):
        print(f"    {s}: {len(v)}")
    
    rng = random.Random(42)
    mmlu_pear_indices = []
    for subj, indices in by_subj.items():
        sampled = rng.sample(indices, min(30, len(indices)))
        mmlu_pear_indices.extend(sampled)
    
    mmlu_pear_source = mmlu_val.select(mmlu_pear_indices, keep_in_memory=True)
else:
    mmlu_pear_source = mmlu_val.select(range(min(1500, len(mmlu_val))), keep_in_memory=True)

mmlu_pear_source.save_to_disk(f"{OUT}/mmlu_pear_source")
print(f"  MMLU PEAR source: {len(mmlu_pear_source)} examples")

# ═══════════════════════════════════════════════════════════════════════
# StrategyQA
# ═══════════════════════════════════════════════════════════════════════
print("\nLoading StrategyQA...")
sqa_train_ds = load_from_disk(f"{BASE}/strategyqa/train")
print(f"  strategyqa/train rows: {len(sqa_train_ds)}")
print(f"  columns: {sqa_train_ds.column_names}")
print(f"  sample: {sqa_train_ds[0]}")

sample_sqa = sqa_train_ds[0]
print(f"  answer type: {type(sample_sqa['answer'])}, value: {sample_sqa['answer']}")

def normalize_sqa(row):
    ans = row['answer']
    if isinstance(ans, bool):
        row['answer_yn'] = "Yes" if ans else "No"
    elif isinstance(ans, str):
        row['answer_yn'] = "Yes" if ans.lower() in ['true', 'yes', '1'] else "No"
    elif isinstance(ans, int):
        row['answer_yn'] = "Yes" if ans == 1 else "No"
    else:
        row['answer_yn'] = "Yes" if str(ans).lower() in ['true', 'yes'] else "No"
    return row

sqa_normalized = sqa_train_ds.map(normalize_sqa, keep_in_memory=True)
sqa_shuffled   = sqa_normalized.shuffle(seed=42, keep_in_memory=True)
sqa_shuffled.save_to_disk(f"{OUT}/sqa_pear_source")
print(f"  StrategyQA PEAR source: {len(sqa_shuffled)}")
print(f"  answer_yn sample: {sqa_shuffled[0]['answer_yn']}")

# ═══════════════════════════════════════════════════════════════════════
# OpenMath
# ═══════════════════════════════════════════════════════════════════════
print("\nLoading OpenMath...")
openmath_path = f"{BASE}/openmath"
contents = os.listdir(openmath_path)
print(f"  openmath contents: {sorted(contents)}")

if "train" in contents:
    openmath_ds = load_from_disk(f"{openmath_path}/train")
else:
    openmath_ds = load_from_disk(openmath_path)

print(f"  openmath rows: {len(openmath_ds)}")
print(f"  columns: {openmath_ds.column_names}")
print(f"  sample keys: {list(openmath_ds[0].keys())}")

openmath_shuffled = openmath_ds.shuffle(seed=42, keep_in_memory=True)

sample_om = openmath_shuffled[0]
sol_field = None
for candidate in ['generated_solution', 'solution', 'answer', 'output']:
    if candidate in sample_om:
        sol_field = candidate
        break
print(f"  Solution field: {sol_field}")
print(f"  Sample solution (first 200 chars): {str(sample_om.get(sol_field,''))[:200]}")

def count_words_om(row):
    sol = row.get(sol_field, '') or ''
    row['sol_words'] = len(sol.split())
    return row

openmath_with_len = openmath_shuffled.map(count_words_om, num_proc=1, keep_in_memory=True)

word_counts = openmath_with_len['sol_words']
short_indices = [i for i, w in enumerate(word_counts) if w <= 150]
hard_indices  = [i for i, w in enumerate(word_counts) if 150 < w <= 400]

print(f"  Short solutions (<=150 words): {len(short_indices)}")
print(f"  Hard solutions (150-400 words): {len(hard_indices)}")

om_pear_indices = short_indices[:min(600, len(short_indices))]
om_grpo_indices = hard_indices[:min(2000, len(hard_indices))]

om_pear = openmath_with_len.select(om_pear_indices, keep_in_memory=True)
om_grpo = openmath_with_len.select(om_grpo_indices, keep_in_memory=True)

om_pear.save_to_disk(f"{OUT}/openmath_pear_source")
om_grpo.save_to_disk(f"{OUT}/openmath_grpo_source")
print(f"  OpenMath PEAR: {len(om_pear)}, GRPO: {len(om_grpo)}")

# ═══════════════════════════════════════════════════════════════════════
# VERIFY
# ═══════════════════════════════════════════════════════════════════════
print("\n=== VERIFICATION ===")
from datasets import load_from_disk as lfd

for name, path in [
    ("gsm_pear_source",      f"{OUT}/gsm_pear_source"),
    ("gsm_grpo_source",      f"{OUT}/gsm_grpo_source"),
    ("mmlu_pear_source",     f"{OUT}/mmlu_pear_source"),
    ("sqa_pear_source",      f"{OUT}/sqa_pear_source"),
    ("openmath_pear_source", f"{OUT}/openmath_pear_source"),
    ("openmath_grpo_source", f"{OUT}/openmath_grpo_source"),
]:
    ds = lfd(path)
    print(f"\n{name}: {len(ds)} rows | cols: {ds.column_names}")
    print(f"  row[0]: {dict(list(ds[0].items())[:4])}")

summary = {
    "gsm_pear":  len(gsm_pear_source),
    "gsm_grpo":  len(gsm_grpo_source),
    "mmlu_pear": len(mmlu_pear_source),
    "sqa_pear":  len(sqa_shuffled),
    "om_pear":   len(om_pear),
    "om_grpo":   len(om_grpo),
}
with open(f"{OUT}/dataset_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))
print("\nNotebook 1 complete. Publish /kaggle/working as dataset: riva-split-datasets")


=== FILE STRUCTURE ===
gsm8k/: ['data-00000-of-00001.arrow', 'dataset_dict.json', 'dataset_info.json', 'state.json', 'test', 'train']
mmlu/: ['auxiliary_train', 'data-00000-of-00001.arrow', 'dataset_dict.json', 'dataset_info.json', 'dev', 'state.json', 'test', 'validation']
strategyqa/: ['data-00000-of-00001.arrow', 'dataset_dict.json', 'dataset_info.json', 'state.json', 'test', 'train']
openmath/: ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']
commonsenseqa/: ['data-00000-of-00001.arrow', 'dataset_dict.json', 'dataset_info.json', 'state.json', 'test', 'train', 'validation']
hotpotqa/: ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']
natural_questions/: ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']

Loading GSM8K...
  gsm8k/train rows: 7473
  columns: ['question', 'answer']
  sample: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

Saving the dataset (1/1 shards): 100%
 3000/3000 [00:00<00:00, 94533.02 examples/s]
Saving the dataset (1/1 shards): 100%
 4473/4473 [00:00<00:00, 137527.74 examples/s]

  GSM8K PEAR: 3000, GRPO: 4473

Loading MMLU validation split...
  mmlu/validation rows: 1531
  columns: ['question', 'subject', 'choices', 'answer']
  sample: {'question': 'The cyclic subgroup of Z_24 generated by 18 has order', 'subject': 'abstract_algebra', 'choices': ['4', '8', '12', '6'], 'answer': 0}
  Subjects found: 57
    college_chemistry: 8
    high_school_computer_science: 9
    global_facts: 10
    abstract_algebra: 11
    business_ethics: 11
    college_computer_science: 11
    college_mathematics: 11
    college_physics: 11
    computer_security: 11
    jurisprudence: 11
    machine_learning: 11
    management: 11
    medical_genetics: 11
    us_foreign_policy: 11
    econometrics: 12
    human_sexuality: 12
    public_relations: 12
    international_law: 13
    anatomy: 14
    formal_logic: 14
    astronomy: 16
    college_biology: 16
    electrical_engineering: 16
    high_school_physics: 17
    high_school_european_history: 18
    logical_fallacies: 18
    virology: 18
    world_religions: 19
    high_school_government_and_politics: 21
    college_medicine: 22
    high_school_chemistry: 22
    high_school_geography: 22
    high_school_us_history: 22
    sociology: 22
    high_school_statistics: 23
    human_aging: 23
    marketing: 25
    conceptual_physics: 26
    high_school_microeconomics: 26
    high_school_world_history: 26
    security_studies: 27
    clinical_knowledge: 29
    high_school_mathematics: 29
    professional_accounting: 31
    professional_medicine: 31
    high_school_biology: 32
    nutrition: 33
    philosophy: 34
    prehistory: 35
    moral_disputes: 38
    elementary_mathematics: 41
    high_school_macroeconomics: 43
    high_school_psychology: 60
    professional_psychology: 69
    miscellaneous: 86
    moral_scenarios: 100
    professional_law: 170

Saving the dataset (1/1 shards): 100%
 1148/1148 [00:00<00:00, 44604.96 examples/s]

  MMLU PEAR source: 1148 examples

Loading StrategyQA...
  strategyqa/train rows: 1603
  columns: ['qid', 'term', 'description', 'question', 'answer', 'facts']
  sample: {'qid': '4fd64bb6ce5b78ab20b6', 'term': 'Mixed martial arts', 'description': 'full contact combat sport', 'question': 'Is Mixed martial arts totally original from Roman Colosseum games?', 'answer': False, 'facts': 'Mixed Martial arts in the UFC takes place in an enclosed structure called The Octagon. The Roman Colosseum games were fought in enclosed arenas where combatants would fight until the last man was standing. Mixed martial arts contests are stopped when one of the combatants is incapacitated. The Roman Colosseum was performed in front of crowds that numbered in the tens of thousands. Over 56,000 people attended UFC 193.'}
  answer type: <class 'bool'>, value: False

Map: 100%
 1603/1603 [00:00<00:00, 18205.74 examples/s]
Saving the dataset (1/1 shards): 100%
 1603/1603 [00:00<00:00, 58630.65 examples/s]

  StrategyQA PEAR source: 1603
  answer_yn sample: Yes

Loading OpenMath...
  openmath contents: ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']
  openmath rows: 40000
  columns: ['problem', 'generated_solution', 'expected_answer', 'problem_source']
  sample keys: ['problem', 'generated_solution', 'expected_answer', 'problem_source']
  Solution field: generated_solution
  Sample solution (first 200 chars): First, we use the fact that $z^2 + z + 1 = 0$. This means that:
\[ z^3 - 1 = (z - 1)(z^2 + z + 1) = 0 \]

Since $z \neq 1$, we have $z^3 = 1$. This implies $z^{96} = 1$.

Now, compute $z^{97} + z^{98}

Map (num_proc=1): 100%
 40000/40000 [00:05<00:00, 11813.04 examples/s]

  Short solutions (<=150 words): 22810
  Hard solutions (150-400 words): 15608

Saving the dataset (1/1 shards): 100%
 600/600 [00:00<00:00, 27831.22 examples/s]
Saving the dataset (1/1 shards): 100%
 2000/2000 [00:00<00:00, 75788.80 examples/s]

  OpenMath PEAR: 600, GRPO: 2000

=== VERIFICATION ===

gsm_pear_source: 3000 rows | cols: ['question', 'answer']
  row[0]: {'question': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?', 'answer': 'Mimi has 2 x 12 = <<2*12=24>>24 sea shells.\nKyle has 24 x 2 = <<24*2=48>>48 sea shells.\nLeigh has 48 / 3 = <<48/3=16>>16 sea shells.\n#### 16'}

gsm_grpo_source: 4473 rows | cols: ['question', 'answer']
  row[0]: {'question': 'Tony has $87. He needs to buy some cheese, which costs $7 a pound and a pound of beef that costs $5 a pound. After buying the beef and his cheese, he has $61 left. How many pounds of cheese did he buy?', 'answer': 'He spent $26 because 87 - 61 = <<87-61=26>>26\nHe spend $21 on cheese because 26 -5 = <<26-5=21>>21\nHe bought 3 pounds of cheese because 21 / 7 = <<21/7=3>>3\n#### 3'}

mmlu_pear_source: 1148 rows | cols: ['question', 'subject', 'choices', 'answer']
  row[0]: {'question': 'Statement 1 | If a group has an element of order 10, then the number of elements of order 10 is divisible by 4. Statement 2 | If m and n are positive integers and phi is the Euler phi function, then phi(mn) = phi(m)phi(n).', 'subject': 'abstract_algebra', 'choices': ['True, True', 'False, False', 'True, False', 'False, True'], 'answer': 1}

sqa_pear_source: 1603 rows | cols: ['qid', 'term', 'description', 'question', 'answer', 'facts', 'answer_yn']
  row[0]: {'qid': 'd82193894aa1c12fbc40', 'term': 'Nissan', 'description': 'Japanese automobile manufacturer', 'question': "Do workers at Nissan's headquarters eat with chopsticks?"}

openmath_pear_source: 600 rows | cols: ['problem', 'generated_solution', 'expected_answer', 'problem_source', 'sol_words']
  row[0]: {'problem': 'Find the inverse of the matrix\n\\[\\begin{pmatrix} 9 & -6 \\\\ -2 & 4 \\end{pmatrix}.\\]If the inverse does not exist, then enter the zero matrix.', 'generated_solution': 'To find the inverse of the matrix $\\begin{pmatrix} 9 & -6 \\\\ -2 & 4 \\end{pmatrix}$, we first need to check if the determinant is non-zero.\n\nThe determinant is\n\\[ \\det \\begin{pmatrix} 9 & -6 \\\\ -2 & 4 \\end{pmatrix} = (9)(4) - (-6)(-2) = 36 - 12 = 24 \\]\n\nSince the determinant is non-zero, the inverse exists.\n\nThe inverse of a $2 \\times 2$ matrix $\\begin{pmatrix} a & b \\\\ c & d \\end{pmatrix}$ is given by\n\\[ \\frac{1}{\\det} \\begin{pmatrix} d & -b \\\\ -c & a \\end{pmatrix} \\]\n\nApplying this formula to our matrix:\n\\[ \\frac{1}{24} \\begin{pmatrix} 4 & 6 \\\\ 2 & 9 \\end{pmatrix} \\]\n\nSo the inverse of the given matrix is:\n\\[ \\frac{1}{24} \\begin{pmatrix} 4 & 6 \\\\ 2 & 9 \\end{pmatrix} = \\boxed{\\begin{pmatrix} 1/6 & 1/4 \\\\ 1/12 & 3/8 \\end{pmatrix}} \\]', 'expected_answer': '\\begin{pmatrix} 1/6 & 1/4 \\\\ 1/12 & 3/8 \\end{pmatrix}', 'problem_source': 'augmented_math'}

openmath_grpo_source: 2000 rows | cols: ['problem', 'generated_solution', 'expected_answer', 'problem_source', 'sol_words']
  row[0]: {'problem': 'Let $z$ be a complex number satisfying $z^2 + z + 1 = 0.$  Compute\n\\[z^{97} + z^{98} + z^{99} + z^{100} + z^{101}.\\]', 'generated_solution': 'First, we use the fact that $z^2 + z + 1 = 0$. This means that:\n\\[ z^3 - 1 = (z - 1)(z^2 + z + 1) = 0 \\]\n\nSince $z \\neq 1$, we have $z^3 = 1$. This implies $z^{96} = 1$.\n\nNow, compute $z^{97} + z^{98} + z^{99} + z^{100} + z^{101}$.\n\nUsing $z^3 = 1$, we can rewrite these powers as:\n\\[ z^{97} = z^{96}z = z \\]\n\\[ z^{98} = z^{96}z^2 = z^2 \\]\n\\[ z^{99} = z^{96}z^3 = 1 \\]\n\\[ z^{100} = z^{96}z^4 = z \\]\n\\[ z^{101} = z^{96}z^5 = z^2 \\]\n\nAdding these up, we get:\n\\[ z^{97} + z^{98} + z^{99} + z^{100} + z^{101} = z + z^2 + 1 + z + z^2 = 2(z^2 + z) + 1 \\]\n\nNow, use $z^2 + z + 1 = 0$ to find $z^2 + z = -1$.\n\nSo,\n\\[ 2(z^2 + z) + 1 = 2(-1) + 1 = -2 + 1 = -1 \\]\n\nThis gives us:\n\\[ z^{97} + z^{98} + z^{99} + z^{100} + z^{101} = \\boxed{-1} \\]', 'expected_answer': '-1', 'problem_source': 'math'}

=== SUMMARY ===
{
  "gsm_pear": 3000,
  "gsm_grpo": 4473,
  "mmlu_pear": 1148,
  "sqa_pear": 1603,
  "om_pear": 600,
  "om_grpo": 2000
}

Notebook 1 complete. Publish /kaggle/working as dataset: riva-split-datasets


PEAR Traces Generation  

In [ ]:
# Standard header (wheels install + paths)
import os, sys, json, re, hashlib, random
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_from_disk
import torch

# --- 1. CONFIGURATION & FIXED PATHS ---
os.environ["WANDB_DISABLED"]       = "true"
os.environ["HF_DATASETS_OFFLINE"]  = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_CACHE"]    = "/kaggle/working/cache"
os.makedirs("/kaggle/working/cache", exist_ok=True)

SPLIT_DATA     = "/kaggle/input/datasets/mahmoodavaram/pear-and-grpo"
BASE_MODEL     = "/kaggle/input/models/mahmoodavaram/qwen2-5-7b/transformers/default/1/qwen2.5-7b"
INSTRUCT_MODEL = "/kaggle/input/models/mahmoodavaram/qwen2-5-7b-instruct/transformers/default/1/qwen2.5-7b-instruct"
OUT            = "/kaggle/working"
BATCH_SIZE     = 4

# --- 2. FEW-SHOT TEMPLATES (Added to prevent NameErrors) ---
GSM8K_FEWSHOT = "Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nAnswer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. #### 6\n\n"
MMLU_FEWSHOT_TEMPLATE = "The following are multiple choice questions (with answers) about {subject}.\n\n"
STRATEGYQA_FEWSHOT = "Answer the following question with Yes or No.\n\n"

# --- 3. LOAD MODELS ---
print("Loading Instruct model (behavior policy πβ)...")
inst_tok = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)
if inst_tok.pad_token is None:
    inst_tok.pad_token = inst_tok.eos_token
inst_model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_MODEL, dtype=torch.bfloat16, device_map="auto"
)
inst_model.eval()

print("Loading Base model (target policy πθ)...")
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16, device_map="auto"
)
base_model.eval()

assert inst_tok.vocab_size == base_tok.vocab_size, "Tokenizer mismatch — stop."

# --- 4. HELPER FUNCTIONS ---
def trace_id(prompt, trace):
    return hashlib.md5((prompt + trace).encode()).hexdigest()[:20]

def get_token_logprobs(model, tokenizer, prompt, trace, max_total=2048):
    full_text = prompt + trace
    full_enc = tokenizer(
        full_text, return_tensors="pt",
        max_length=max_total, truncation=True
    ).to(model.device)
    prompt_enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_total)
    prompt_len = prompt_enc['input_ids'].shape[1]
    
    with torch.no_grad():
        out = model(**full_enc)
        logits = out.logits[0]   
    
    log_probs = torch.log_softmax(logits, dim=-1)
    token_ids = full_enc['input_ids'][0]
    seq_len = token_ids.shape[0]
    
    result = []
    for t in range(prompt_len, seq_len):
        lp = log_probs[t-1, token_ids[t]].item()
        result.append(lp)
    return result

def generate_traces(prompts, model, tokenizer, max_new_tokens, temperature, n=1):
    tokenizer.padding_side = "left"
    inputs = tokenizer(
        prompts, return_tensors="pt", truncation=True,
        max_length=2048, padding=True
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.95,
            do_sample=(temperature != 1.0 or n > 1),
            num_return_sequences=n,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05
        )
    prompt_len = inputs['input_ids'].shape[1]
    traces_flat = [
        tokenizer.decode(outputs[i][prompt_len:], skip_special_tokens=True).strip()
        for i in range(len(prompts) * n)
    ]
    return [traces_flat[i*n:(i+1)*n] for i in range(len(prompts))]

def extract_gsm_answer_from_trace(trace):
    if "####" in trace:
        raw = trace.split("####")[1].strip().split("\n")[0]
        nums = raw.split()
        return nums[0].replace(",", "").rstrip(".,;") if nums else None
    return None

def extract_yesno_from_trace(trace):
    m = re.search(r"\b(Yes|No)\b", trace, re.IGNORECASE)
    return m.group(1).capitalize() if m else None

def extract_abcd_from_trace(trace):
    m = re.search(r"Answer:\s*([ABCD])", trace, re.IGNORECASE)
    if m: return m.group(1).upper()
    m = re.search(r"^([ABCD])\b", trace.strip())
    if m: return m.group(1).upper()
    return None

# --- 5. PROCESS GSM8K ---
gsm_pear = load_from_disk(f"{SPLIT_DATA}/gsm_pear_source")
print(f"\nProcessing {len(gsm_pear)} GSM8K PEAR examples...")

gsm_records = []
for batch_start in range(0, len(gsm_pear), BATCH_SIZE):
    batch = gsm_pear.select(range(batch_start, min(batch_start+BATCH_SIZE, len(gsm_pear))))
    
    prompts = []
    golds = []
    for row in batch:
        gold = extract_gsm_answer_from_trace(row['answer'])
        if gold is None:
            m = re.search(r"####\s*([\d,\.]+)", row['answer'])
            gold = m.group(1).replace(",","") if m else None
        golds.append(gold)
        prompt = (GSM8K_FEWSHOT + f"Question: {row['question']}\nAnswer:")
        prompts.append(prompt)
    
    traces_per_prompt = generate_traces(prompts, inst_model, inst_tok, max_new_tokens=400, temperature=0.8, n=3)
    
    for i, (prompt, gold, traces) in enumerate(zip(prompts, golds, traces_per_prompt)):
        got_positive = False
        for trace in traces:
            pred = extract_gsm_answer_from_trace(trace)
            is_correct = (pred is not None and pred == str(gold))
            
            if is_correct and not got_positive:
                b_lps = get_token_logprobs(inst_model, inst_tok, prompt, trace)
                t_lps = get_token_logprobs(base_model, base_tok, prompt, trace)
                gsm_records.append({
                    'id': trace_id(prompt, trace), 'prompt': prompt, 'trace': trace,
                    'gold': gold, 'pred': pred, 'source': 'gsm8k', 'role': 'positive',
                    'behavior_logprobs': b_lps, 'base_logprobs': t_lps
                })
                got_positive = True
            elif not is_correct and len(trace.split('.')) >= 3:
                b_lps = get_token_logprobs(inst_model, inst_tok, prompt, trace)
                t_lps = get_token_logprobs(base_model, base_tok, prompt, trace)
                gsm_records.append({
                    'id': trace_id(prompt, trace), 'prompt': prompt, 'trace': trace,
                    'gold': gold, 'pred': pred, 'source': 'gsm8k', 'role': 'negative',
                    'behavior_logprobs': b_lps, 'base_logprobs': t_lps
                })
    
    if (batch_start // BATCH_SIZE) % 25 == 0:
        with open(f"{OUT}/gsm_traces_partial.jsonl", 'w') as f:
            for r in gsm_records:
                f.write(json.dumps(r) + '\n')
        pos = sum(1 for r in gsm_records if r['role']=='positive')
        neg = sum(1 for r in gsm_records if r['role']=='negative')
        print(f"  GSM8K {batch_start+BATCH_SIZE}/{len(gsm_pear)} — pos: {pos}, neg: {neg}")

with open(f"{OUT}/gsm_traces.jsonl", 'w') as f:
    for r in gsm_records: f.write(json.dumps(r) + '\n')

# --- 6. PROCESS MMLU ---
letter_map = {0:'A', 1:'B', 2:'C', 3:'D'}
mmlu_pear = load_from_disk(f"{SPLIT_DATA}/mmlu_pear_source")
print(f"\nProcessing {len(mmlu_pear)} MMLU PEAR examples...")

mmlu_records = []
for batch_start in range(0, len(mmlu_pear), BATCH_SIZE):
    batch = mmlu_pear.select(range(batch_start, min(batch_start+BATCH_SIZE, len(mmlu_pear))))
    
    prompts = []
    golds = []
    for row in batch:
        gold_letter = letter_map[row['answer']]
        golds.append(gold_letter)
        subj = row.get('subject', 'general knowledge').replace('_', ' ')
        choices_text = '\n'.join([f"{letter_map[i]}) {row['choices'][i]}" for i in range(4)])
        prompt = (MMLU_FEWSHOT_TEMPLATE.format(subject=subj) +
                  f"Question: {row['question']}\n{choices_text}\nAnswer:")
        prompts.append(prompt)
    
    traces_per_prompt = generate_traces(prompts, inst_model, inst_tok, max_new_tokens=200, temperature=0.3, n=3)
    
    for i, (prompt, gold, traces) in enumerate(zip(prompts, golds, traces_per_prompt)):
        got_positive = False
        for trace in traces:
            pred = extract_abcd_from_trace(trace)
            is_correct = (pred == gold)
            if is_correct and not got_positive:
                b_lps = get_token_logprobs(inst_model, inst_tok, prompt, trace)
                t_lps = get_token_logprobs(base_model, base_tok, prompt, trace)
                mmlu_records.append({
                    'id': trace_id(prompt, trace), 'prompt': prompt, 'trace': trace,
                    'gold': gold, 'pred': pred, 'subject': batch[i].get('subject',''),
                    'source': 'mmlu', 'role': 'positive',
                    'behavior_logprobs': b_lps, 'base_logprobs': t_lps
                })
                got_positive = True
    
    if (batch_start // BATCH_SIZE) % 25 == 0:
        pos = sum(1 for r in mmlu_records if r['role']=='positive')
        print(f"  MMLU {batch_start+BATCH_SIZE}/{len(mmlu_pear)} — pos: {pos}")

with open(f"{OUT}/mmlu_traces.jsonl", 'w') as f:
    for r in mmlu_records: f.write(json.dumps(r) + '\n')

# --- 7. PROCESS STRATEGYQA ---
sqa_pear = load_from_disk(f"{SPLIT_DATA}/sqa_pear_source")
print(f"\nProcessing {len(sqa_pear)} StrategyQA PEAR examples...")

sqa_records = []
for batch_start in range(0, len(sqa_pear), BATCH_SIZE):
    batch = sqa_pear.select(range(batch_start, min(batch_start+BATCH_SIZE, len(sqa_pear))))
    
    prompts = []
    golds = []
    for row in batch:
        gold = row['answer_yn']
        golds.append(gold)
        desc = row.get('description', '') or ''
        term = row.get('term', '') or ''
        prompt = (STRATEGYQA_FEWSHOT +
                  f"Q: {row['question']}\nContext: {term} — {desc}\nA:")
        prompts.append(prompt)
    
    traces_per_prompt = generate_traces(prompts, inst_model, inst_tok, max_new_tokens=150, temperature=0.8, n=3)
    
    for i, (prompt, gold, traces) in enumerate(zip(prompts, golds, traces_per_prompt)):
        got_positive = False
        for trace in traces:
            pred = extract_yesno_from_trace(trace)
            is_correct = (pred == gold)
            if is_correct and not got_positive:
                b_lps = get_token_logprobs(inst_model, inst_tok, prompt, trace)
                t_lps = get_token_logprobs(base_model, base_tok, prompt, trace)
                sqa_records.append({
                    'id': trace_id(prompt, trace), 'prompt': prompt, 'trace': trace,
                    'gold': gold, 'pred': pred, 'source': 'strategyqa', 'role': 'positive',
                    'behavior_logprobs': b_lps, 'base_logprobs': t_lps
                })
                got_positive = True
            elif not is_correct and pred is not None:
                b_lps = get_token_logprobs(inst_model, inst_tok, prompt, trace)
                t_lps = get_token_logprobs(base_model, base_tok, prompt, trace)
                sqa_records.append({
                    'id': trace_id(prompt, trace), 'prompt': prompt, 'trace': trace,
                    'gold': gold, 'pred': pred, 'source': 'strategyqa', 'role': 'negative',
                    'behavior_logprobs': b_lps, 'base_logprobs': t_lps
                })
    
    if (batch_start // BATCH_SIZE) % 25 == 0:
        pos = sum(1 for r in sqa_records if r['role']=='positive')
        neg = sum(1 for r in sqa_records if r['role']=='negative')
        print(f"  SQA {batch_start+BATCH_SIZE}/{len(sqa_pear)} — pos: {pos}, neg: {neg}")

with open(f"{OUT}/sqa_traces.jsonl", 'w') as f:
    for r in sqa_records: f.write(json.dumps(r) + '\n')

# --- 8. PROCESS OPENMATH ---
om_pear = load_from_disk(f"{SPLIT_DATA}/openmath_pear_source")
print(f"\nProcessing {len(om_pear)} OpenMath PEAR examples...")

om_records = []
for batch_start in range(0, len(om_pear), BATCH_SIZE):
    batch = om_pear.select(range(batch_start, min(batch_start+BATCH_SIZE, len(om_pear))))
    prompts = []
    for row in batch:
        prompt = (GSM8K_FEWSHOT.replace("####","Answer:") + f"Question: {row['problem']}\nAnswer:")
        prompts.append(prompt)
    
    traces_per_prompt = generate_traces(prompts, inst_model, inst_tok, max_new_tokens=250, temperature=0.8, n=1)
    for i, (prompt, traces) in enumerate(zip(prompts, traces_per_prompt)):
        trace = traces[0]
        b_lps = get_token_logprobs(inst_model, inst_tok, prompt, trace)
        t_lps = get_token_logprobs(base_model, base_tok, prompt, trace)
        om_records.append({
            'id': trace_id(prompt, trace), 'prompt': prompt, 'trace': trace,
            'gold': batch[i]['expected_answer'], 'source': 'openmath', 'role': 'positive',
            'behavior_logprobs': b_lps, 'base_logprobs': t_lps
        })

with open(f"{OUT}/om_traces.jsonl", 'w') as f:
    for r in om_records: f.write(json.dumps(r) + '\n')

# --- 9. FINAL DATASET ASSEMBLY ---
print("\nAssembling final dataset...")
random.seed(42)

def load_jsonl(path):
    with open(path) as f: return [json.loads(l) for l in f]

gsm_all  = load_jsonl(f"{OUT}/gsm_traces.jsonl")
mmlu_all = load_jsonl(f"{OUT}/mmlu_traces.jsonl")
sqa_all  = load_jsonl(f"{OUT}/sqa_traces.jsonl")
om_all   = load_jsonl(f"{OUT}/om_traces.jsonl")

gsm_pos = [r for r in gsm_all if r['role']=='positive']
gsm_neg = [r for r in gsm_all if r['role']=='negative']
sqa_pos = [r for r in sqa_all if r['role']=='positive']
sqa_neg = [r for r in sqa_all if r['role']=='negative']

gsm_neg_capped = random.sample(gsm_neg, min(len(gsm_pos)//2, len(gsm_neg)))
sqa_neg_capped = random.sample(sqa_neg, min(len(sqa_pos)//2, len(sqa_neg)))

final_dataset = gsm_pos + gsm_neg_capped + mmlu_all + sqa_pos + sqa_neg_capped + om_all
random.shuffle(final_dataset)

with open(f"{OUT}/pear_sft_final.jsonl", 'w') as f:
    for r in final_dataset: f.write(json.dumps(r) + '\n')

stats = {
    "total": len(final_dataset),
    "gsm_pos": len(gsm_pos), "gsm_neg": len(gsm_neg_capped),
    "mmlu_pos": len(mmlu_all),
    "sqa_pos": len(sqa_pos), "sqa_neg": len(sqa_neg_capped),
    "om_pos": len(om_all)
}
with open(f"{OUT}/trace_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("✅ RUN COMPLETE. Final Statistics:")
print(json.dumps(stats, indent=2))

4.7s	1	0.00s - Debugger warning: It seems that frozen modules are being used, which may
4.7s	2	0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
4.7s	3	0.00s - to python to disable frozen modules.
4.7s	4	0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
5.0s	5	0.00s - Debugger warning: It seems that frozen modules are being used, which may
5.0s	6	0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
5.0s	7	0.00s - to python to disable frozen modules.
5.0s	8	0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
23.1s	9	Loading Instruct model (behavior policy πβ)...
118.8s	10	Loading Base model (target policy πθ)...
212.0s	11	
212.0s	12	Processing 3000 GSM8K PEAR examples...
222.2s	13	  GSM8K 4/3000 — pos: 3, neg: 8
440.3s	14	  GSM8K 104/3000 — pos: 85, neg: 134
658.4s	15	  GSM8K 204/3000 — pos: 170, neg: 251
877.5s	16	  GSM8K 304/3000 — pos: 252, neg: 382
1096.9s	17	  GSM8K 404/3000 — pos: 334, neg: 510
1316.7s	18	  GSM8K 504/3000 — pos: 412, neg: 656
1534.1s	19	  GSM8K 604/3000 — pos: 499, neg: 765
1754.1s	20	  GSM8K 704/3000 — pos: 580, neg: 908
1974.7s	21	  GSM8K 804/3000 — pos: 660, neg: 1049
2193.9s	22	  GSM8K 904/3000 — pos: 743, neg: 1178
2414.3s	23	  GSM8K 1004/3000 — pos: 823, neg: 1326
2634.7s	24	  GSM8K 1104/3000 — pos: 907, neg: 1466
2854.8s	25	  GSM8K 1204/3000 — pos: 985, neg: 1607
3075.1s	26	  GSM8K 1304/3000 — pos: 1064, neg: 1754
3295.5s	27	  GSM8K 1404/3000 — pos: 1140, neg: 1897
3515.6s	28	  GSM8K 1504/3000 — pos: 1219, neg: 2041
3735.1s	29	  GSM8K 1604/3000 — pos: 1306, neg: 2162
3955.5s	30	  GSM8K 1704/3000 — pos: 1384, neg: 2302
4175.0s	31	  GSM8K 1804/3000 — pos: 1467, neg: 2431
4396.1s	32	  GSM8K 1904/3000 — pos: 1546, neg: 2579
4615.1s	33	  GSM8K 2004/3000 — pos: 1626, neg: 2707
4837.3s	34	  GSM8K 2104/3000 — pos: 1707, neg: 2859
5057.4s	35	  GSM8K 2204/3000 — pos: 1786, neg: 2999
5278.4s	36	  GSM8K 2304/3000 — pos: 1865, neg: 3142
5498.6s	37	  GSM8K 2404/3000 — pos: 1947, neg: 3273
5718.3s	38	  GSM8K 2504/3000 — pos: 2028, neg: 3401
5938.3s	39	  GSM8K 2604/3000 — pos: 2109, neg: 3535
6159.0s	40	  GSM8K 2704/3000 — pos: 2188, neg: 3663
6379.1s	41	  GSM8K 2804/3000 — pos: 2269, neg: 3792
6599.1s	42	  GSM8K 2904/3000 — pos: 2356, neg: 3909
6809.6s	43	
6809.6s	44	Processing 1148 MMLU PEAR examples...
6813.4s	45	  MMLU 4/1148 — pos: 0
6911.9s	46	  MMLU 104/1148 — pos: 65
7011.1s	47	  MMLU 204/1148 — pos: 122
7109.9s	48	  MMLU 304/1148 — pos: 177
7213.0s	49	  MMLU 404/1148 — pos: 249
7311.8s	50	  MMLU 504/1148 — pos: 309
7421.5s	51	  MMLU 604/1148 — pos: 388
7519.9s	52	  MMLU 704/1148 — pos: 461
7617.9s	53	  MMLU 804/1148 — pos: 528
7716.5s	54	  MMLU 904/1148 — pos: 589
7821.8s	55	  MMLU 1004/1148 — pos: 650
7924.1s	56	  MMLU 1104/1148 — pos: 721
7967.6s	57	
7967.6s	58	Processing 1603 StrategyQA PEAR examples...
7970.6s	59	  SQA 4/1603 — pos: 3, neg: 3
8045.6s	60	  SQA 104/1603 — pos: 69, neg: 102
8120.2s	61	  SQA 204/1603 — pos: 140, neg: 188
8195.5s	62	  SQA 304/1603 — pos: 205, neg: 294
8269.9s	63	  SQA 404/1603 — pos: 270, neg: 385
8344.9s	64	  SQA 504/1603 — pos: 338, neg: 484
8419.5s	65	  SQA 604/1603 — pos: 404, neg: 574
8494.9s	66	  SQA 704/1603 — pos: 467, neg: 682
8569.2s	67	  SQA 804/1603 — pos: 540, neg: 761
8643.8s	68	  SQA 904/1603 — pos: 605, neg: 853
8719.2s	69	  SQA 1004/1603 — pos: 668, neg: 965
8793.7s	70	  SQA 1104/1603 — pos: 738, neg: 1055
8868.6s	71	  SQA 1204/1603 — pos: 803, neg: 1153
8944.8s	72	  SQA 1304/1603 — pos: 859, neg: 1288
9019.8s	73	  SQA 1404/1603 — pos: 924, neg: 1390
9095.0s	74	  SQA 1504/1603 — pos: 988, neg: 1494
9169.5s	75	  SQA 1604/1603 — pos: 1055, neg: 1585
9169.8s	76	
9169.8s	77	Processing 600 OpenMath PEAR examples...
9850.3s	78	
9850.3s	79	Assembling final dataset...
9852.1s	80	✅ RUN COMPLETE. Final Statistics:
9852.1s	81	{
9852.1s	82	  "total": 6584,
9852.1s	83	  "gsm_pos": 2435,
9852.1s	84	  "gsm_neg": 1217,
9852.1s	85	  "mmlu_pos": 750,
9852.1s	86	  "sqa_pos": 1055,
9852.1s	87	  "sqa_neg": 527,
9852.1s	88	  "om_pos": 600
9852.1s	89	}
9856.2s	90	/usr/local/lib/python3.12/dist-packages/mistune.py:435: SyntaxWarning: invalid escape sequence '\|'
9856.2s	91	  cells[i][c] = re.sub('\\\\\|', '|', cell)
9856.3s	92	/usr/local/lib/python3.12/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'
9856.3s	93	  text = re.sub(r'_', '\_', text) # Escape underscores in display text
9856.6s	94	[NbConvertApp] Converting notebook __notebook__.ipynb to notebook
9856.8s	95	[NbConvertApp] Writing 47866 bytes to __notebook__.ipynb
9857.9s	96	[NbConvertApp] Converting notebook __notebook__.ipynb to html
9858.3s	97	[NbConvertApp] Writing 375271 bytes to __results__.html

PEAR SFT Training

In [ ]:
import os
import sys
import json
import re
import hashlib
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader

# --- 1. CONFIGURATION & FIXED PATHS ---
os.environ["WANDB_DISABLED"]       = "true"
os.environ["HF_DATASETS_OFFLINE"]  = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_CACHE"]    = "/kaggle/working/cache"
os.makedirs("/kaggle/working/cache", exist_ok=True)

SPLIT_DATA     = "/kaggle/input/datasets/mahmoodavaram/pear-and-grpo"
BASE_MODEL     = "/kaggle/input/models/mahmoodavaram/qwen2-5-7b/transformers/default/1/qwen2.5-7b"
OUT            = "/kaggle/working"
TRACES_PATH    = "/kaggle/input/datasets/mahmoodavaram/fpeargrpo"

# --- 2. HYPERPARAMETERS (PEAR Paper §4.2) ---
GAMMA         = 0.999
LOG_G_MIN     = -10.0
LOG_G_MAX     = 5.0
LOG_DELTA_MIN = -0.08
LOG_DELTA_MAX = 0.3
LR            = 1e-5
EPOCHS        = 1

# --- 3. HARDWARE OPTIMIZATIONS (RTX Pro 6000) ---
BATCH_SIZE    = 4        # Parallel execution across GPU cores
GRAD_ACCUM    = 4        # 4 * 4 = 16 effective batch size
NEG_LAMBDA    = 1.0      # Repulsion coefficient for negative traces
MAX_SEQ_LEN   = 1024

random.seed(42)
torch.manual_seed(42)

# --- 4. INITIALIZE COMPONENTS ---
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Base Model into VRAM...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, 
    dtype=torch.bfloat16,  # Fixed deprecation warning
    device_map="cuda"
)
model.train()

# --- 5. DATASET & LOGIC DEFINITION ---
class PEARDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_len):
        self.tok = tokenizer
        self.max_len = max_len
        self.items = []
        with open(jsonl_path) as f:
            for line in f:
                r = json.loads(line)
                b_lps = r.get('behavior_logprobs', [])
                t_lps = r.get('base_logprobs', [])
                if len(b_lps) > 0 and len(t_lps) > 0:
                    self.items.append(r)
        print(f"PEARDataset: {len(self.items)} valid full production examples loaded.")
    
    def __len__(self): 
        return len(self.items)
    
    def __getitem__(self, idx):
        item = self.items[idx]
        prompt = item['prompt']
        trace  = item['trace']
        
        # Robust fallback for key naming mismatches
        role = item.get('role', item.get('training_role', 'positive'))
        
        full_enc   = self.tok(prompt + trace, max_length=self.max_len, truncation=True, return_tensors='pt')
        prompt_enc = self.tok(prompt, max_length=self.max_len, truncation=True, return_tensors='pt')
        
        input_ids  = full_enc['input_ids'][0]
        prompt_len = prompt_enc['input_ids'].shape[1]
        trace_len  = len(input_ids) - prompt_len
        
        if trace_len <= 0:
            return {
                'input_ids': input_ids, 'prompt_len': prompt_len,
                'pear_weights': torch.zeros(max(1, trace_len)),
                'role': role
            }
        
        T = min(trace_len, len(item['behavior_logprobs']), len(item['base_logprobs']))
        b_lps = item['behavior_logprobs'][:T]
        t_lps = item['base_logprobs'][:T]
        
        clipped_deltas = [
            max(LOG_DELTA_MIN, min(LOG_DELTA_MAX, t_lps[j] - b_lps[j]))
            for j in range(T)
        ]
        
        log_gamma = torch.tensor(GAMMA).log().item()
        pear_weights = torch.zeros(T)
        suffix_sum = 0.0
        for t in reversed(range(T)):
            log_g = (T - 1 - t) * log_gamma + suffix_sum
            log_g_clipped = max(LOG_G_MIN, min(LOG_G_MAX, log_g))
            pear_weights[t] = torch.exp(torch.tensor(log_g_clipped)).item()
            suffix_sum += clipped_deltas[t]
        
        return {
            'input_ids': input_ids,
            'prompt_len': prompt_len,
            'pear_weights': pear_weights,
            'role': role
        }

def pear_collate(batch):
    max_len = max(x['input_ids'].shape[0] for x in batch)
    input_ids_pad = torch.zeros(len(batch), max_len, dtype=torch.long)
    attn_mask     = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, x in enumerate(batch):
        L = x['input_ids'].shape[0]
        input_ids_pad[i, :L] = x['input_ids']
        attn_mask[i, :L] = 1
    return {
        'input_ids': input_ids_pad,
        'attention_mask': attn_mask,
        'items': batch
    }

# --- 6. DATA LOADER & TRACKING SETUP ---
dataset    = PEARDataset(f"{TRACES_PATH}/pear_sft_final.jsonl", tokenizer, MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pear_collate)

total_steps = len(dataloader) * EPOCHS
optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.05 * total_steps // GRAD_ACCUM)),
    num_training_steps=max(2, total_steps // GRAD_ACCUM)
)

print(f"Total Rows: {len(dataset)} | Total Batches: {len(dataloader)} | Effective Step Target: {total_steps // GRAD_ACCUM}")
print("Starting Production PEAR SFT Training Loop...")

optimizer.zero_grad()
running_loss = 0.0
print_interval = 200  # Track and report progress every 200 complete optimizer updates

# --- 7. TRAINING LOOP ---
for step, batch in enumerate(dataloader):
    input_ids = batch['input_ids'].to('cuda')
    attn_mask = batch['attention_mask'].to('cuda')
    
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        outputs = model(input_ids=input_ids, attention_mask=attn_mask)
        logits  = outputs.logits
    
    batch_loss = 0.0
    valid_items_in_batch = 0
    
    # Surgical unpadded loss calculation per item in the batch
    for i, item in enumerate(batch['items']):
        prompt_len = item['prompt_len']
        weights    = item['pear_weights'].to('cuda')
        role       = item['role']
        
        trace_start = prompt_len
        trace_end   = item['input_ids'].shape[0]
        T = trace_end - trace_start
        
        if T <= 0:
            continue
            
        trace_logits  = logits[i, trace_start-1:trace_end-1, :]
        trace_targets = input_ids[i, trace_start:trace_end]
        
        log_probs     = torch.log_softmax(trace_logits, dim=-1)
        token_nll     = -log_probs[range(T), trace_targets[:T]]
        
        w = weights[:T]
        
        if role == 'positive':
            loss_element = (w.detach() * token_nll).mean()
        else:
            # Repulsive loss implementation for negative alignment paths
            seq_w = w.mean().detach()
            loss_element = -NEG_LAMBDA * seq_w * token_nll.mean()
            
        batch_loss += loss_element
        valid_items_in_batch += 1
        
    if valid_items_in_batch == 0:
        continue
        
    loss = (batch_loss / valid_items_in_batch) / GRAD_ACCUM
    loss.backward()
    running_loss += loss.item() * GRAD_ACCUM
    
    # Optimizer Execution Step
    if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(dataloader):
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        update_step = (step + 1) // GRAD_ACCUM
        if update_step % print_interval == 0 or update_step == 1:
            avg_loss = running_loss / print_interval if update_step > 1 else running_loss
            print(f"Update Step {update_step}/{total_steps//GRAD_ACCUM} | Metrics Loss: {avg_loss:.4f}")
            if update_step > 1:
                running_loss = 0.0

# --- 8. SAVE ARTIFACTS ---
checkpoint_dir = f"{OUT}/pear_sft_checkpoint"
model.save_pretrained(checkpoint_dir)
tokenizer.save_pretrained(checkpoint_dir)
print(f"PEAR SFT full model training finished. Checkpoint saved to: {checkpoint_dir}")
print("DO NOT EVAL THIS STEP — Offline performance matching baseline is expected pipeline behavior.")

with open(f"{OUT}/pear_sft_training_complete.json", "w") as f:
    json.dump({"status": "complete", "note": "Do not eval mid-pipeline"}, f)

Time
	
#
	
Log Message
4.4s	1	0.00s - Debugger warning: It seems that frozen modules are being used, which may
4.4s	2	0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
4.4s	3	0.00s - to python to disable frozen modules.
4.4s	4	0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
4.7s	5	0.00s - Debugger warning: It seems that frozen modules are being used, which may
4.7s	6	0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
4.7s	7	0.00s - to python to disable frozen modules.
4.7s	8	0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
20.7s	9	Loading Tokenizer...
21.1s	10	Loading Base Model into VRAM...
151.3s	11	PEARDataset: 6584 valid full production examples loaded.
151.3s	12	Total Rows: 6584 | Total Batches: 1646 | Effective Step Target: 411
151.3s	13	Starting Production PEAR SFT Training Loop...
154.0s	14	Update Step 1/411 | Metrics Loss: 1.0172
520.2s	15	Update Step 200/411 | Metrics Loss: -2.4449
885.6s	16	Update Step 400/411 | Metrics Loss: -22.1698
919.1s	17	PEAR SFT full model training finished. Checkpoint saved to: /kaggle/working/pear_sft_checkpoint
919.1s	18	DO NOT EVAL THIS STEP — Offline performance matching baseline is expected pipeline behavior.
919.1s	19	Files inside directory '1':
919.1s	20	['cache', 'pear_sft_training_complete.json', 'pear_sft_checkpoint', '__notebook__.ipynb']
923.7s	21	/usr/local/lib/python3.12/dist-packages/mistune.py:435: SyntaxWarning: invalid escape sequence '\|'
923.7s	22	  cells[i][c] = re.sub('\\\\\|', '|', cell)
923.8s	23	/usr/local/lib/python3.12/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'
923.8s	24	  text = re.sub(r'_', '\_', text) # Escape underscores in display text
944.3s	25	[NbConvertApp] Converting notebook __notebook__.ipynb to notebook
944.4s	26	[NbConvertApp] Writing 38456 bytes to __notebook__.ipynb
945.6s	27	[NbConvertApp] Converting notebook __notebook__.ipynb to html
946.0s	28	[NbConvertApp] Writing 334603 bytes to __results__.html
